In [39]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [40]:
# Reading The dataframe from csv file 

df = pd.read_csv('data/stud.csv')

df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [41]:
# Preparing X and Y Variables 

X = df.drop('math_score',axis=1)
Y = df['math_score']
 
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [42]:
# Seperating numerical attributes and categorical attributes

num_attribs = X.select_dtypes(exclude='object').columns
cat_attribs = X.select_dtypes(include='object').columns


In [43]:
# Creating pipeline to perform One Hot Encoding , Ordinal Encoding and Standardization on the features
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

def create_pipeline(num_attribs,complex_cat_attribs,simple_cat_attribs):
    num_pipeline = Pipeline([
        ('Standardization',StandardScaler())
    ])
    complex_cat_pipeline = Pipeline([
        ('OnehotEncoding',OneHotEncoder())
    ])
    simple_cat_pipeline = Pipeline([
        ('OrdinalEncoder',OrdinalEncoder())
    ])
    
    final_pipeline = ColumnTransformer([
        ('num_pipeline',num_pipeline,num_attribs),
        ('cat_pipeline1',complex_cat_pipeline,complex_cat_attribs),
        ('cat_pipeline2',simple_cat_pipeline,simple_cat_attribs)
    ])
    
    return final_pipeline

In [44]:
# Seperating Simple and Complex categorical attributes 
complex_cat_attribs = [column for column in cat_attribs if len(X[column].unique())>2]
simple_cat_attribs = [column for column in cat_attribs if len(X[column].unique())<=2]

In [45]:
pipeline = create_pipeline(num_attribs,complex_cat_attribs,simple_cat_attribs)


In [46]:
# Transforming X into transformed data
from sklearn.model_selection import train_test_split

X_train , X_test , Y_train , Y_test = train_test_split(X,Y,test_size=0.25,random_state=42)

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

X_train.shape

(750, 16)

In [47]:
# Test Statistic
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = root_mean_squared_error(true, predicted)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [48]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, Y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(Y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(Y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.2972
- Mean Absolute Error: 4.2383
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4825
- Mean Absolute Error: 4.3379
- R2 Score: 0.8778


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5469
- Mean Absolute Error: 5.1796
- R2 Score: 0.8080
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.6500
- Mean Absolute Error: 5.2184
- R2 Score: 0.8202


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.2977
- Mean Absolute Error: 4.2370
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.4801
- Mean Absolute Error: 4.3357
- R2 Score: 0.8779


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 6.0975
- Mean Absolute Error: 4.9048
- R2 Score: 0.8334
-----------------------

In [49]:
# Checking Results 
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
2,Ridge,0.877933
0,Linear Regression,0.877824
7,CatBoosting Regressor,0.855749
5,Random Forest Regressor,0.848074
8,AdaBoost Regressor,0.842098
6,XGBRegressor,0.837134
1,Lasso,0.820248
3,K-Neighbors Regressor,0.778164
4,Decision Tree,0.774379
